#### <b>EE6332 - Modelling and Optimization in VLSI - Final Project</b>
<b>Continuous and Discrete Gate sizing</b>
--------------------------------------------------------------------------------
Author: Aman Saini (ee24s007)





Paths can be edited based on the location of the required files, and where the results are required to be saved.

In [40]:
from pylab import *
import networkx as nx, matplotlib.pyplot as plt
from networkx.readwrite import json_graph
from tabulate import tabulate
import contextlib, re, os, sys
import cvxpy as cp
import numpy as np

# path where the discrete sizing results are required to be saved
folder_path_discrete = "/content/drive/MyDrive/EE6332/discrete_solutions"
# path to the json files storing the DAG representing the circuit
base_path_json = "/content/drive/My Drive/EE6332/DAG_json"
# path, where the continuous sizing results are required to be saved
path_continuous = "/content/drive/MyDrive/EE6332/sizing_results"
# sizes_dict stores the sizes obtained of each gate-type after discretization
sizes_dict_path = "/content/drive/MyDrive/EE6332/"
# mosek license
mosek_license = "/content/drive/MyDrive/EE6332/mosek.lic"

If prompted to restart the runtime, choose 'Cancel'.

In [ ]:
!curl https://download.mosek.com/stable/8.1.0.82/mosektoolslinux64x86.tar.bz2 --output /root/mosektoolslinux64x86.tar.bz2
!cd /root/; tar xvf /root/mosektoolslinux64x86.tar.bz2
import shutil
# shutil.rmtree('/content/drive', ignore_errors=True)
from google.colab import drive
drive.mount('/content/drive')
!cp -avr "$mosek_license" /root/mosek/
!pip install git+https://github.com/convexengineering/gpkit
!pip show gpkit
from collections import defaultdict

import gpkit, json, time, os, math
from gpkit import VectorVariable, Variable, Model, units

# --- Set up MOSEK paths ---
os.environ["MOSEKLM_LICENSE_FILE"] = "/root/mosek/mosek.lic"  # License file path
os.environ["MOSEK_HOME"] = "/root/mosektoolslinux64x86"       # MOSEK installation path
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["MOSEK_HOME"], "bin")
os.environ["LD_LIBRARY_PATH"] = (
    os.path.join(os.environ["MOSEK_HOME"], "tools", "platform", "linux64x86", "bin")
    + os.pathsep
    + os.environ.get("LD_LIBRARY_PATH", "")
)

# --- Install solvers ---
!pip install --upgrade mosek
# Step 1: Install build dependencies
!apt-get install -y pkg-config libblas-dev liblapack-dev gfortran

!pip uninstall -y cvxpy mosek
!pip install cvxpy[mosek]

# --- Import and test ---
import mosek
print("MOSEK version:", mosek.Env().getversion())

# !pip uninstall -y cvxpy cplex
# !pip install cvxpy[cplex]

# Show available solvers to confirm ECOS_BB is available
print("Installed solvers:", cp.installed_solvers())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 40.3M  100 40.3M    0     0  62.2M      0 --:--:-- --:--:-- --:--:-- 62.2M
mosek/8/tools/examples/python/parameters.py
mosek/8/toolbox/examples/cqo1.m
mosek/8/toolbox/r2014a/mskqpopt.m
mosek/8/tools/examples/fusion/java/breaksolver.java
mosek/8/toolbox/r2014a/mskcheck.m
mosek/8/tools/examples/fusion/cxx/lo1.cc
mosek/8/tools/examples/data/dgo.f
mosek/8/tools/examples/dotnet/scopt1.cs
mosek/8/tools/examples/dotnet/solvebasis.cs
mosek/8/toolbox/r2014aom/mskerrmsg.m
mosek/8/toolbox/examples/simple.m
mosek/8/tools/examples/java/solutionquality.java
mosek/8/tools/examples/fusion/dotnet/production.cs
mosek/8/tools/examples/fusion/java/Makefile
mosek/8/tools/examples/fusion/python/lo1.py
mosek/8/tools/examples/fusion/java/duality.java
mosek/8/tools/platform/linux64x86/bin/libcilkrts.so.5
mosek/8/toolbox/examples/qo2.m
mosek/8/tools/ex

MOSEK version: (11, 0, 20)
Installed solvers: ['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'HIGHS', 'MOSEK', 'OSQP', 'SCIPY', 'SCS']


### Reading the DAG in the JSON file format.
Parser reads the circuit netlist, and constructs the Directed Acyclic Graph (DAG) using the networkx library. The DAG can be dumped in the form of a JSON file, which can be read by the opimizer program.

In [24]:
class CircuitGraph:
    def __init__(self, ckt_file, base_path=base_path_json):
        """
        Initializes the circuit graph by loading from a JSON file.
        Adds indices to nodes and builds gate info map.
        """
        self.ckt_file = ckt_file
        self.json_path = f"{base_path}/dag_{ckt_file}.json"
        self.G = self._load_graph()
        self._assign_node_indices()
        self.gate_info_map = self._build_gate_info_map()

    def _load_graph(self):
        """
        Load the JSON graph file and return a NetworkX graph object.
        """
        with open(self.json_path, "r") as f:
            data = json.load(f)
        return json_graph.node_link_graph(data, directed=True, multigraph=False, edges="links")

    def _assign_node_indices(self):
        """
        Assign a unique index to each node (n_idx), and a gate index (g_idx) if not input.
        """
        for idx, node in enumerate(self.G.nodes()):
            self.G.nodes[node]['n_idx'] = idx

        gate_idx = 0
        for node in self.G.nodes():
            node_data = self.G.nodes[node]
            if node_data['type'] != 'input':
                node_data['g_idx'] = gate_idx
                gate_idx += 1
            else:
                node_data['g_idx'] = -1

    def _build_gate_info_map(self):
        """
        Build a dictionary mapping gate names to their type and index.
        """
        gate_info = {}
        for node in self.G.nodes():
            attr = self.G.nodes[node]
            if attr['type'] != 'input':
                gate_info[attr['name']] = (attr['type'], attr['g_idx'])
        return gate_info

    def get_graph(self):
        """
        Returns the graph object.
        """
        return self.G

    def get_gate_info_map(self):
        """
        Returns the gate info mapping: {gate_name: (gate_type, gate_idx)}
        """
        return self.gate_info_map

### <b>Phase 1: Continuous geometric program for gate sizing</b>
###### Formulating the sizing problem in the form of a GP
  - The number of variables = 2 * num(gates) + num(inputs)
    - (gate sizes) & (arrival time at all nodes)
  - 1 <= gate_size <= max_gate_size
  - arrival time(inputs) = 1 {as the arrival time = 0 will not be a valid GP constraint}

In [25]:
class GateSizingModel:
    def __init__(self, G, gate_info_map):
        """
        Initialize vector variables, delay parameters, and other constants for gate sizing.
        """
        self.G = G
        self.gate_info_map = gate_info_map

        self.n_gates = len(gate_info_map)
        self.n_nodes = len(G.nodes)

        # Gate size vector variable (size of each gate)
        self.g_size = VectorVariable(self.n_gates, 'x')

        # Arrival time at all circuit nodes
        self.a_t = VectorVariable(self.n_nodes, 'a_t')

        # Delay contribution for each fanout gate
        self.delay_dict = self._create_delay_dict()

        # T0 represents the overall circuit delay (i.e., Twall)
        self.T0 = Variable('T0')

        # Delay models
        self.pd_dict = self._init_parasitic_delays()
        self.le_dict = self._init_logical_efforts()

    def _create_delay_dict(self):
        """
        Create delay dictionary with vector variables for fanout delays.
        """
        delay_dict = {}
        idx_gate = 0
        for node, attr in self.G.nodes(data=True):
            if attr.get('type') != 'input':
                fanouts = list(self.G.successors(node))
                if len(fanouts) > 0:
                    delay_dict[node] = VectorVariable(len(fanouts), f"d_{idx_gate}")
                else:
                    delay_dict[node] = []
                idx_gate += 1
        return delay_dict

    def _init_parasitic_delays(self):
        """
        Returns a dictionary of parasitic delays for each gate type.
        """
        pd_dict = {"NOT": 1}
        for i in range(2, 5):
            pd_dict[f"NAND_{i}"] = i
            pd_dict[f"NOR_{i}"] = i
        return pd_dict

    def _init_logical_efforts(self):
        """
        Returns a dictionary of logical efforts for each gate type.
        """
        le_dict = {"NOT": 1}
        for i in range(2, 5):
            le_dict[f"NAND_{i}"] = (i + 2) / 3
            le_dict[f"NOR_{i}"] = (2 * i + 1) / 3
        return le_dict

    def get_variables(self):
        """
        Return all GP variables and constants as a dictionary.
        """
        return {
            'g_size': self.g_size,
            'a_t': self.a_t,
            'delay_dict': self.delay_dict,
            'T0': self.T0,
            'pd_dict': self.pd_dict,
            'le_dict': self.le_dict,
            'out_cap': self.out_cap,
            'max_in_cap': self.max_in_cap,
            'max_gate_size': self.max_gate_size
        }

#### <b>Setting up the GP problem by adding the appropriate constraints</b>

 - Sizing Constraints
   - maximim and minimum sizes
   - maximum input gate sizes
 - Input arrival times (set to 1, as monomial == 0 is not a valid GP constraint)
 - Set <b>Twall</b> greater than all the output arrival times
 - To handle all the fan-out gates, a dictionary of vector variables can be used:
   - the delay contributed by each fan-out gate is accumulated using the vector variable corresponding to a particular gate.

In [26]:
class ConstraintBuilder:
    """
    This class builds the constraints for the gate sizing problem.
    """
    def __init__(self, G, g_size, T0, a_t, delay_dict, gate_info_map,
                 le_dict, pd_dict, out_cap, max_gate_size, max_in_cap):
        self.G = G
        self.g_size = g_size
        self.T0 = T0
        self.a_t = a_t
        self.delay_dict = delay_dict
        self.gate_info_map = gate_info_map
        self.le_dict = le_dict
        self.pd_dict = pd_dict
        self.out_cap = out_cap
        self.max_gate_size = max_gate_size
        self.max_in_cap = max_in_cap
        self.constraints = []

    # GATE SIZE CONSTRAINTS
    def gate_size_constraints(self):
        """
        constraints related to gate sizes (1 <= gate_size <= max_gate_size)
        constraints related to input gate sizes (sum(input gate_size) <= max_in_cap)
        """
        # (size should be between the allowed limits) (1 <= size <= max_gate_size)
        for n in range(len(self.g_size)):
            self.constraints.append(self.g_size[n] >= 1)
            self.constraints.append(self.g_size[n] <= self.max_gate_size)

        # For each primary input, limit the total load it drives (sum of le * size of each gate it connects to)
        for node, attr in self.G.nodes(data=True):
            if attr.get('type') == 'input':
                total_load = 0
                for succ in self.G.successors(node):  # all gates driven by this input
                    gate_type, gate_idx = self.gate_info_map[self.G.nodes[succ]['name']]
                    le = self.le_dict[gate_type]
                    total_load += le * self.g_size[gate_idx]
                self.constraints.append(total_load <= self.max_in_cap)

    # TIMING CONSTRAINTS (I/O TIMINGS)

    def timing_constraints(self):
        """
        constraints related to arrival time at primary inputs and primary outputs
        """
        for node, attr in self.G.nodes(data=True):
            # Set the arrival time at INPUTS = 1
            if attr.get('type') == 'input':
                self.constraints.append(self.a_t[attr['n_idx']] == 1)
                continue

            # Set T0 >= arrival time at OUTPUTS
            if len(list(self.G.successors(node))) == 0:
                self.constraints.append(self.T0 >= self.a_t[attr['n_idx']])

    # (ADDING GATE DELAYS TO TIMING CONSTRAINTS)
    def delay_constraints(self):
        """
        constraints related to gate delays. Relating the arrival time at the output of a gate to the inputs using gate delays.
        """
        G = self.G
        for node, attr in G.nodes(data=True):
            curr_gate = attr.get('name')

            if attr.get('type') != 'input':  # current node is the output of a gate
                pd = self.pd_dict[G.nodes[node]['type']]

                # directly getting the required fan-in and fan-out lists from the graph
                fan_in_nodes = list(G.predecessors(node))  # all fan-in nodes (by node ID)
                fan_out_gates = [G.nodes[succ]['name'] for succ in G.successors(node) if G.nodes[succ]['type'] != 'input']

                fo_len = len(fan_out_gates)

                for idx, gate in enumerate(fan_out_gates):
                    le = self.le_dict[self.gate_info_map[gate][0]]
                    if idx != 0: # accumulating delay along the vector variable
                        self.constraints.append(
                            self.delay_dict[node][idx] >= le * self.g_size[self.gate_info_map[gate][1]] / self.g_size[self.gate_info_map[curr_gate][1]] + self.delay_dict[node][idx - 1]
                        )
                    else:
                        self.constraints.append(
                            self.delay_dict[node][idx] >= le * self.g_size[self.gate_info_map[gate][1]] / self.g_size[self.gate_info_map[curr_gate][1]]
                        )

                if not fan_out_gates:
                    # a primary output, with no gates connected to it (only driving the output capacitance)
                    if(attr.get('is_output'), False):
                        for fi_node in fan_in_nodes:
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd + self.out_cap / self.g_size[self.gate_info_map[curr_gate][1]])
                    else:
                        for fi_node in fan_in_nodes:
                        # no fan-out gates, and not a primary output (FLOATING GATE). (In this case, only parasitic delay is there)
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd)


                else:
                    for fi_node in fan_in_nodes:
                        if attr.get('is_output', False) and (self.out_cap != 0): # if the gate is marked as an output gate, it is driving the output capacitance in addition to fan-out gates (so additional delay term)
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd + self.delay_dict[node][fo_len - 1] + self.out_cap / self.g_size[self.gate_info_map[curr_gate][1]])
                        else:
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd + self.delay_dict[node][fo_len - 1])

    def build_constraints(self):
        """
        builds the constraints (sizing constraints, timing constraints, delay constraints)
        """
        self.gate_size_constraints()
        self.timing_constraints()
        self.delay_constraints()
        return self.constraints


#### <b>Objective function</b>
 - for Twall, if T0 is given as objective function, all the output arrival times are fixed to Twall by gpkit. Even if the arrival times corresponding to the sizing solution is less.
 - to counter this the following objective function is used:
   - <b>Objective = T0 + weight_factor * sum(a_t(outputs))</b>
   - <b>weight_factor</b> needs to be set to some low value, such that it does not disturb the Twall solution, and corresponding sizes.
 - Similar approach is also followed for area-minimum solution for a given Tspec.

In [27]:
class TSolver:
    """
    Gate sizing solver for digital circuits.

    This class builds a circuit graph, sets up a convex optimization problem for either
    minimum delay (Twall) or minimum area with a timing constraint (Tspec), and solves it.

    Args:
        ckt_name (str): Name of the benchmark circuit.
        CircuitGraphClass: Class to generate the circuit DAG from a netlist.
        GateSizingModelClass: Provides gate size, delay, and load expressions.
        ConstraintBuilderClass: Builds all sizing and timing constraints.
        constants (tuple):
            - (out_cap, max_in_cap, max_gate_size) for delay minimization
            - (out_cap, max_in_cap, max_gate_size, Tspec, relax_factor) for area minimization

    Usage:
        solver = TSolver('c432', CircuitGraph, GateSizingModel, ConstraintBuilder, (1000, 50, 64))
        solver.solve()

    Outputs:
        - solver.solution: Optimization result
        - solver.Twall: Minimum delay (if solved for delay)
        - solver.sizes: Gate sizes from the solution
    """
    def __init__(self, ckt_name, CircuitGraphClass, GateSizingModelClass, ConstraintBuilderClass, constants):
        self.ckt_name = ckt_name
        self.CircuitGraphClass = CircuitGraphClass
        self.GateSizingModelClass = GateSizingModelClass
        self.ConstraintBuilderClass = ConstraintBuilderClass
        if(len(constants) == 3):
            self.out_cap, self.max_in_cap, self.max_gate_size = constants
            self.Tspec = -1
            self.relax_factor = -1
        elif(len(constants) == 5):
            self.out_cap, self.max_in_cap, self.max_gate_size, self.Tspec, self.relax_factor = constants

        self.sizes = None
        self.Twall = None

        self._initialize()

    def _initialize(self):
        # Build the circuit graph and extract basic data
        self.solution = None
        self.ckt = self.CircuitGraphClass(self.ckt_name)
        self.G = self.ckt.get_graph()
        self.gate_info_map = self.ckt.get_gate_info_map()

        # Initialize sizing model
        self.sizing_model = self.GateSizingModelClass(self.G, self.gate_info_map)
        self.g_size = self.sizing_model.g_size
        self.a_t = self.sizing_model.a_t
        self.T0 = self.sizing_model.T0
        self.delay_dict = self.sizing_model.delay_dict
        self.le_dict = self.sizing_model.le_dict
        self.pd_dict = self.sizing_model.pd_dict

        # Build constraints
        self.constraints = self.ConstraintBuilderClass(
            self.G, self.g_size, self.T0, self.a_t, self.delay_dict,
            self.gate_info_map, self.le_dict, self.pd_dict,
            self.out_cap, self.max_gate_size, self.max_in_cap
        ).build_constraints()

    def solve(self):
        """
        Solves the optimization problem and returns key results.
        """
        outputs = []
        for node, attr in self.G.nodes(data=True):
            if attr.get('is_output', False):
                outputs.append(attr['n_idx'])
        if(self.Tspec == -1):
            # objective = self.a_t[0]
            objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)
        else:
            # minimize the area with a certain output timing specification
            objective = sum(self.g_size)# + 0.0001 * sum(self.a_t[i] for i in outputs)
            self.constraints.append(self.T0 <= self.Tspec)

        model = Model(objective, self.constraints)
        start_time = time.time()
        self.solution = model.solve(verbosity=1)
        elapsed_time = time.time() - start_time

        print(f"Total elapsed time (including setup): {elapsed_time:.4f} seconds")

# TEMPORARY CONDITION CHECK ----------------------------------------------------
# Check if arrival times of all the outputs are fixed to one value

        # print("\033[1mArrival time at outputs:\033[0m")
        # for node, attr in self.G.nodes(data=True):
        #     if attr.get('is_output', False):
        #         print(attr.get('name'), attr.get('n_idx'))
        #         t_out = self.solution["variables"]["a_t"][attr.get('n_idx')]
        #         print(f"Arrival time: {(t_out-1)}")

# Conclusion: Need to mention the arrival time of outputs in the objective fn
# otherwise, they are ceiled to Twall, even if their arrival time is lower.
#
# ------------------------------------------------------------------------------
# Objective = T0 + weight_factor * np.sum(a_t[outputs])
# ------------------------------------------------------------------------------
# weight factor should be low to not affect Twall result

def save_sizing_results(solver, folder_path, relax_factor=None):
    """
    Save the sizing results from the solver into a file inside the given folder.
    The results include the gate index (as the index), the node name, its size, and its name.

    Parameters:
    - solver: an object with `ckt_name`, `constants`, `G` (DAG), and `solution['variables']['x']`.
    - folder_path: where to store the result file.
    - relax_factor: used in file naming if optimization is minA.
    """

    # Create folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    # Define file name
    if solver.Tspec == -1:
        file_name = f"{solver.ckt_name}_minT_.txt"
    else:
        file_name = f"{solver.ckt_name}_minA_{relax_factor}.txt"

    file_path = os.path.join(folder_path, file_name)

    total_size = 0
    # Gather gate sizing data
    gate_data = []
    for node, attr in solver.G.nodes(data=True):
        g_idx = attr.get('g_idx', -1)
        if g_idx != -1:
            node_name = node  # Use the node name as node_name
            gate_name = attr.get('name', 'Unknown')  # Default to 'Unknown' if no name is provided
            size = round(solver.solution['variables']['x'][g_idx], 3)
            total_size += size
            gate_data.append((g_idx, node_name, gate_name, size))  # Store gate_index first

    # Create table headers and format with tabulate
    table = tabulate(gate_data, headers=["Gate Index", "Node Name", "Gate Name", "Gate Size"], tablefmt="grid")

    # Write to file
    with open(file_path, 'w') as f:
        f.write("Sizing Results:\n")
        f.write(f"Circuit name: {solver.ckt_name}\n\n")
        f.write(f"T0: {(solver.solution['variables']['T0'] - 1):.3f}\n")
        f.write(f"Sum of gate sizes: {total_size:.3f}\n\n")
        f.write(table)

    print(f"Results saved to: {file_path}\n")

#### <b>1, 2. Twall, and Area minimum solution (for a given T_spec)</b>
 - the solver class takes the following inputs:
   - circuit name
   - output capacitance
   - maximum input capacitance
   - maximum gate size possible
 - subject to above design constants, minimum delay (T_wall), and minimum area solution (A_min) is obtained.

In [28]:
sys.setrecursionlimit(100000)
### --- RESULT DUMP CODE BLOCK ---

sizing_constants = (1000, 50, 64)  # (out_cap, max_in_cap, max_gate_size) (Normalized to 1x inverter size)
ckt = "c17"
folder_path = f"{path_continuous}/{ckt}"
# Relaxed timing constraint (Tspec = Twall * relax_factor) (AREA+OPTIMIZATION)
relax_factor = 1.5
relax_factor_arr = [1.3]

# Solver for Twall
solver = TSolver(
    ckt_name=ckt,
    CircuitGraphClass=CircuitGraph,
    GateSizingModelClass=GateSizingModel,
    ConstraintBuilderClass=ConstraintBuilder,
    constants=sizing_constants
)

solver.solve()
Twall = solver.solution["variables"]["T0"]
sizes = list(solver.solution["variables"]["x"])
sizes_rounded = ['%.3f' % val for val in sizes]
sum_sizes = sum(sizes)


print(f"\n\033[1mTwall = {(Twall - 1):.3f}\033[0m\n")
print("Sizes of gates =", sizes_rounded)
print("Sum of sizes:", round(sum_sizes, 2))
# Saving sizing results
save_sizing_results(solver, folder_path=folder_path, relax_factor=relax_factor)
print("===================================================================================")

for relax_factor in relax_factor_arr:
    # solver for Opimal area, given a timing specification
    sizing_constants_A = sizing_constants + (relax_factor*(Twall - 1) + 1, relax_factor)
    solver_A = TSolver(
        ckt_name=ckt,
        CircuitGraphClass=CircuitGraph,
        GateSizingModelClass=GateSizingModel,
        ConstraintBuilderClass=ConstraintBuilder,
        constants=sizing_constants_A
    )

    solver_A.solve()
    Tspec_A = solver_A.solution["variables"]["T0"]
    sizes_A = list(solver_A.solution["variables"]["x"])
    sizes_rounded_A = ['%.2f' % val for val in sizes_A]
    sum_sizes_A = sum(sizes_A)

    print(f"\nT_delay (Area Optimized; Tspec = {relax_factor}x Twall) = {(Tspec_A - 1):.3f}")
    print("Sizes of gates =", sizes_rounded_A)
    print(f"\033[1mSum of sizes: {round(sum_sizes_A, 2)}\033[0m")

    # Save sizing results
    save_sizing_results(solver_A, folder_path=folder_path, relax_factor=relax_factor)
    print("===================================================================================")

<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)


Using solver 'mosek_cli'
 for 24 free variables
  in 48 posynomial inequalities.
Solving took 0.15 seconds.
Total elapsed time (including setup): 0.1754 seconds

Twall = 28.833

Sizes of gates = ['9.268', '28.232', '37.500', '18.750', '64.000', '64.000']
Sum of sizes: 221.75
Results saved to: /content/drive/MyDrive/EE6332/sizing_results/c17/c17_minT_.txt

Using solver 'mosek_cli'
 for 24 free variables
  in 49 posynomial inequalities.
Solving took 0.087 seconds.
Total elapsed time (including setup): 0.1098 seconds

T_delay (Area Optimized; Tspec = 1.3x Twall) = 37.482
Sizes of gates = ['5.28', '14.07', '17.30', '8.65', '45.40', '45.40']
Sum of sizes: 136.11
Results saved to: /content/drive/MyDrive/EE6332/sizing_results/c17/c17_minA_1.3.txt



In [29]:
# UTILITY FUNCTION TO GET THE SOLUTION FOR A GIVEN CIRCUIT, FOR A GIVEN TSPEC
def get_solution(ckt_name, relax_factor = -1):
    """
    Solves the continuous gate sizing problem for a given circuit.

    This function first computes the minimum possible delay (Twall) for the given circuit.
    If a relax_factor is provided, it then solves a second problem to minimize area
    while meeting a relaxed timing specification (Tspec = 1 + relax_factor * (Twall - 1)).

    Args:
        ckt_name (str): Name of the ISCAS85 benchmark circuit (e.g., 'c432').
        relax_factor (float, optional):
            If -1 (default), only delay is minimized (Twall).
            If set, area is minimized with a relaxed timing constraint.

    Returns:
        TSolver: A TSolver instance containing the solution (either delay- or area-optimal).

    Usage:
        solver = get_solution('c432')               # Get delay-optimal solution
        solver_A = get_solution('c432', 1.2)        # Get area-optimal solution with relaxed Tspec
    """
    sizing_constants = (1000, 50, 64)  # (out_cap, max_in_cap, max_gate_size) (Normalized to 1x inverter size)
    # Solver for Twall
    folder_path = f"{path_continuous}/{ckt}"
    solver = TSolver(
        ckt_name=ckt_name,
        CircuitGraphClass=CircuitGraph,
        GateSizingModelClass=GateSizingModel,
        ConstraintBuilderClass=ConstraintBuilder,
        constants=sizing_constants
    )

    solver.solve()
    if(relax_factor == -1):
        return solver
    Twall = solver.solution['variables']['T0']
    # get the optimal area solution
    sizing_constants_A = sizing_constants + (relax_factor*(Twall - 1) + 1, relax_factor)
    solver_A = TSolver(
        ckt_name=ckt_name,
        CircuitGraphClass=CircuitGraph,
        GateSizingModelClass=GateSizingModel,
        ConstraintBuilderClass=ConstraintBuilder,
        constants=sizing_constants_A
    )

    solver_A.solve()
    return solver_A

#### <b>3. Critical path analysis</b>

 - Analysis of gate sizes on a critical path & non-critical path.
 - Get the slack of each node in the graph. The critical path can be found by traversing from input to output along a path with all zero slacks.
 - Non-critical path will have some nodes associated with non-zero slacks.
   - for area-minimum solutions, the gates with a positive slack are expected to have size = 1.0

In [30]:
def get_critical_path(solver):
    G = solver.G
    a_t = solver.solution['variables']['a_t']
    T_delay = solver.solution['variables']['T0'] - 1

    # Step 1: Compute gate delays
    gate_delays = {}
    for node in G.nodes:
        node_data = G.nodes[node]
        if node_data['type'] != 'input':
            out_time = a_t[node_data['n_idx']]
            in_times = [a_t[G.nodes[pred]['n_idx']] for pred in G.predecessors(node)]
            gate_delays[node] = out_time - max(in_times)

    # Step 2: Initialize RATs
    RAT = {node: None for node in G.nodes}
    outputs = [node for node in G.nodes if G.nodes[node].get('is_output', False)]
    for node in outputs:
        RAT[node] = T_delay

    # Step 3: Backward propagate RATs
    for node in reversed(list(nx.topological_sort(G))):
        if G.nodes[node]['type'] != 'input':
            for pred in G.predecessors(node):
                delay = gate_delays[node]
                new_rat = RAT[node] - delay
                if RAT[pred] is None:
                    RAT[pred] = new_rat
                else:
                    RAT[pred] = min(RAT[pred], new_rat)

    # Step 4: Compute slacks====================================================
    slack = {}
    for node in G.nodes:
        arrival = a_t[G.nodes[node]['n_idx']] - 1
        slack[node] = abs(RAT[node] - arrival)
        if(slack[node] < 1e-6):
            slack[node] = 0

    # Step 5: Find critical path (zero slack path)==============================
    critical_path = []
    for input_node in nx.topological_sort(G):
        if G.nodes[input_node]['type'] != 'input':
            continue
        if abs(slack[input_node]) > 1e-6:
            continue

        path = [input_node]
        current = input_node
        while not G.nodes[current].get('is_output', False):
            zero_slack_children = [
                succ for succ in G.successors(current)
                if abs(slack[succ]) < 1e-6
            ]
            if not zero_slack_children:
                break
            current = max(zero_slack_children, key=lambda n: a_t[G.nodes[n]['n_idx']])
            path.append(current)

        if len(path) > len(critical_path):
            critical_path = path

    # Step 6: Find non-critical path (positive slack input)=====================
    non_critical_path = []
    for input_node in nx.topological_sort(G):
        if G.nodes[input_node]['type'] != 'input':
            continue
        if slack[input_node] <= 0:
            continue  # Not a valid non-critical input

        path = [input_node]
        current = input_node
        while not G.nodes[current].get('is_output', False):
            candidates = list(G.successors(current))
            if not candidates:
                break

            # Prefer positive slack successors
            pos_slack_succs = [s for s in candidates if slack[s] > 0]
            if pos_slack_succs:
                next_node = max(pos_slack_succs, key=lambda n: a_t[G.nodes[n]['n_idx']])
            else:
                # If no positive slack successors, allow zero slack
                zero_slack_succs = [s for s in candidates if abs(slack[s]) < 1e-6]
                if not zero_slack_succs:
                    break
                next_node = max(zero_slack_succs, key=lambda n: a_t[G.nodes[n]['n_idx']])

            current = next_node
            path.append(current)

        if len(path) > len(non_critical_path):
            non_critical_path = path
    return slack, critical_path, non_critical_path



def print_slacks(solver):
    slacks, critical_path, non_critical_path = get_critical_path(solver)
    G = solver.G

    critical_slacks = []
    non_critical_slacks = []

    print_order = list(nx.topological_sort(G))  # Ensures input nodes appear first

    for idx, node_name in enumerate(print_order, start=1):
        if node_name not in slacks:
            continue  # Skip if no slack info (may not happen, but safe)

        slack = slacks[node_name]
        slack_rounded = round(slack, 3)
        if slack_rounded < 1e-6:
            slack_rounded = 0.0

        node_data = G.nodes[node_name]
        gate_name = node_data['name']
        gate_size = round(solver.solution['variables']['x'][node_data['g_idx']], 2) if 'g_idx' in node_data else '_'

        if node_data['type'] == 'input':
            node_type = 'Input'
            gate_name = '_'
            gate_size = '_'
        elif node_data.get('is_output', False):
            node_type = 'Output'
        else:
            node_type = ' '

        row = [idx, node_name, gate_name, gate_size, node_type, slack_rounded]

        if node_name in critical_path:
            critical_slacks.append(row)
        if node_name in non_critical_path:
            non_critical_slacks.append(row)

    print("--- Solution Details ---")
    print(f"ISCAS85 Benchmark ckt: {solver.ckt_name}")

    if solver.Tspec == -1:
        print(f"T_delay: {(solver.solution['variables']['T0'] - 1):.3f}; (Twall Solution)")
    else:
        print(f"T_delay: {(solver.solution['variables']['T0'] - 1):.3f}; (Area Optimized; T_spec = {(solver.Tspec - 1):.3f})")

    # Print critical path slacks
    print(f"\n--- Slack for Critical Path (Topological Order) ---")
    print(tabulate(
        critical_slacks,
        headers=["S.No", "Node", "Gate", "Size", "Node Type", "Slack"],
        tablefmt="pretty",
        numalign="right",
        stralign="center"
    ))

    # Print non-critical path slacks
    print(f"\n--- Slack for Non-Critical Path (Topological Order) ---")
    print(tabulate(
        non_critical_slacks,
        headers=["S.No", "Node", "Gate", "Size", "Node Type", "Slack"],
        tablefmt="pretty",
        numalign="right",
        stralign="center"
    ))

print_slacks(solver_A)

--- Solution Details ---
ISCAS85 Benchmark ckt: c17
T_delay: 37.482; (Area Optimized; T_spec = 37.482)

--- Slack for Critical Path (Topological Order) ---
+------+------+---------+-------+-----------+-------+
| S.No | Node |  Gate   | Size  | Node Type | Slack |
+------+------+---------+-------+-----------+-------+
|  4   |  N6  |    _    |   _   |   Input   |  0.0  |
|  7   | N11  | NAND2_2 | 14.07 |           |  0.0  |
|  8   | N16  | NAND2_3 | 17.3  |           |  0.0  |
|  10  | N22  | NAND2_5 | 45.4  |  Output   |  0.0  |
+------+------+---------+-------+-----------+-------+

--- Slack for Non-Critical Path (Topological Order) ---
+------+------+---------+------+-----------+-------+
| S.No | Node |  Gate   | Size | Node Type | Slack |
+------+------+---------+------+-----------+-------+
|  2   |  N2  |    _    |  _   |   Input   | 4.46  |
|  8   | N16  | NAND2_3 | 17.3 |           |  0.0  |
|  10  | N22  | NAND2_5 | 45.4 |  Output   |  0.0  |
+------+------+---------+------+-----

### <b>4. Discretizing the continuous solution </b>


#### <b>4.1. Using Heuristics</b>
The following gate size discretization schemes based on heuristics can be applied, and their performance evaluated to start with the discretization problem.

 - <b>Floor</b> the sizes of the input gates.
 - <b>Ceil</b> the sizes of the remaining gates.
 - <b>Round</b> the gate sizes to the nearest integer.
 - To verify that the <b>T_delay</b> for discretized solutions is correct, the continuous solution is also reevaluated using new algorithm, which can be matched with the solution by GPkit.

In [31]:
# Helper function to compute T_delay for a given gate size assignment
def get_delay_from_sizes(solver, sizes_dict):

    """
    Computes the total delay (T_delay) for a given gate size assignment in the circuit.

    Args:
        solver (TSolver): The solver instance containing the circuit graph and parameters.
        sizes_dict (dict): Dictionary mapping each node to its assigned gate size.

    Returns:
        float: The maximum input-to-output delay computed based on the given sizes.

    Description:
        Calculates gate delays using output capacitance and intrinsic delay,
        then computes arrival times in a topological order to find the critical path delay.
    """

    tmp_G = solver.G
    gate_delays = {}

    # Get the delay of each gate in the circuit
    for node in tmp_G.nodes:
        node_data = tmp_G.nodes[node]
        if node_data['type'] != 'input':
            out_cap = 0
            for succ in tmp_G.successors(node):
                le = solver.le_dict[tmp_G.nodes[succ]['type']]
                out_cap += le * sizes_dict[succ]
            if tmp_G.nodes[node].get('is_output', False):
                out_cap += 1000
            gate_delays[node] = out_cap / sizes_dict[node] + solver.pd_dict[node_data['type']]

    # Compute arrival times (moving topologically from input towards the output)
    a_t_discrete = {}
    T_delay_D = 0
    for node in nx.topological_sort(tmp_G):
        node_data = tmp_G.nodes[node]
        if node_data['type'] == 'input':
            a_t_discrete[node] = 0
        else:
            in_times = max(a_t_discrete[pred] for pred in tmp_G.predecessors(node))
            a_t_discrete[node] = in_times + gate_delays[node]
            T_delay_D = max(T_delay_D, a_t_discrete[node])

    return T_delay_D

In [32]:

# Main function for discretization
def _get_heuristic_discretization(solver):
    """
    Generates and evaluates discrete approximations of the continuous gate sizing solution.

    Args:
        solver (TSolver): A solver instance that already contains a continuous solution.

    Returns:
        list of dict: A list of 4 dictionaries, each mapping gate nodes to their sizes:
                      [floor, ceil, round, continuous].

    Description:
        This function applies floor, ceil, and round discretization to the continuous gate sizes
        and compares their delays and area (sum of sizes) against the original continuous solution.
        Prints benchmark info and delay/area metrics for each variant.
    """
    discrete_solutions = [{}, {}, {}, {}]
    discrete_solutions_key = ["floor", "ceil", "round", "continuous"]
    tmp_G = solver.G
    sum_sizes = [0, 0, 0, 0]

    for node in tmp_G.nodes:
        node_data = tmp_G.nodes[node]
        if node_data['type'] != 'input':
            size = solver.solution['variables']['x'][node_data['g_idx']]
            discrete_solutions[0][node] = math.floor(size)
            discrete_solutions[1][node] = math.ceil(size)
            discrete_solutions[2][node] = round(size)
            discrete_solutions[3][node] = size
            for i in range(4):
                sum_sizes[i] += discrete_solutions[i][node]

    # Print continuous solution details
    Tspec_C = solver.solution["variables"]["T0"]
    sizes_C = list(solver.solution["variables"]["x"])
    sum_sizes_C = round(np.sum(sizes_C), 2)
    print("--- Continuous Solution Details ---")
    print("ISCAS85 Benchmark ckt:", solver.ckt_name)
    if solver.Tspec == -1:
        print(f"\033[1mT_delay: {(Tspec_C - 1):.3f}\033[0m --->  (Twall Solution)")
    else:
        print(f"\033[1mT_delay: {(Tspec_C - 1):.3f}\033[0m --->  [Optimal Area Solution, (T_spec/T_wall) = {solver.relax_factor}]")
    print(f"Sum of sizes: {sum_sizes_C}\n")

    # Evaluate all discrete solutions
    for idx, sizes_dict in enumerate(discrete_solutions):
        # if(idx == 3):
        #     continue
        T_delay_D = get_delay_from_sizes(solver, sizes_dict)
        print(f"--- Discrete Solution Details; Type: ({discrete_solutions_key[idx]}) ---")
        print(f"\033[1mT_delay: {T_delay_D:.3f}\033[0m")
        print(f"Sum of sizes: {sum_sizes[idx]:.2f}\n")

    return discrete_solutions


discrete_solutions = _get_heuristic_discretization(solver_A)

--- Continuous Solution Details ---
ISCAS85 Benchmark ckt: c17
T_delay: 37.482 --->  [Optimal Area Solution, (T_spec/T_wall) = 1.3]
Sum of sizes: 136.11

--- Discrete Solution Details; Type: (floor) ---
T_delay: 38.222
Sum of sizes: 134.00

--- Discrete Solution Details; Type: (ceil) ---
T_delay: 36.954
Sum of sizes: 140.00

--- Discrete Solution Details; Type: (round) ---
T_delay: 38.222
Sum of sizes: 135.00

--- Discrete Solution Details; Type: (continuous) ---
T_delay: 37.482
Sum of sizes: 136.11



#### <b>4.2. Branch and Bound</b>
- The following code block implements a BFS variant of the branch and bound algorithm
- Each depth level represents a discretization step for a single gate in the circuit.
- The best <b>'k'</b> solutions can be explored further from every level, and the other branches can be bounded. (In this case the best solution signifies the least T_delay)
- For each of the best 'k' solution nodes in the previous depth level, we can get two new solution nodes in the current level, obtained from flooring and ceiling the current gate size.

In [55]:
def branch_nd_bound(solver, srch_idx = 8):

    """
    Performs branch-and-bound based discretization of gate sizes for a given circuit.

    This function explores discrete gate sizing solutions by rounding continuous sizes
    using a BFS-style traversal with a configurable search width (`srch_idx`).
    It then refines the best discrete solution with local tweaks and finally performs
    area optimization under timing constraints.

    Args:
        solver: An object containing the circuit graph, continuous solution, and utility methods.
        srch_idx (int): Number of top candidates to explore at each level during BFS (default: 8).

    Returns:
        Tuple[float, dict]: Final timing delay and a dictionary mapping gate names to discrete sizes.
    """

    print(f"--- Beginning discretization for {solver.ckt_name}; T_spec = T_wall*{1 if solver.relax_factor == -1 else solver.relax_factor} ---")
    start_time = time.time()
    tmp_G = solver.G
    cont_sol_dict = {}
    sum_sizes_cont = 0

    for node in tmp_G.nodes:
        node_data = tmp_G.nodes[node]
        if(node_data['type'] != 'input'):
            cont_sol_dict[node] = solver.solution['variables']['x'][node_data['g_idx']]
            sum_sizes_cont += cont_sol_dict[node]

    T_cont = solver.solution['variables']['T0'] - 1 # delay (continuous gate sizing)

    # perform BFS, explore maximum of 'srch_idx' branches on every level

    curr_level = [] # stores the solutions at the current level
    curr_level.append((T_cont, cont_sol_dict))

    for node in tmp_G.nodes:
        node_data = tmp_G.nodes[node]
        if(node_data['type'] != 'input'):
            curr_level = sorted(curr_level, key = lambda x: x[0]) # sorting the solutions in curr level based on the T_delay
            nxt_level = []
            for i in range(min(srch_idx, len(curr_level))):
                tmp_sol_new = curr_level[i][1]
                if(abs(tmp_sol_new[node] - math.floor(tmp_sol_new[node])) < 1e-3):
                    # if the gate size is already close to an integer value, just round the gate size and continue
                    tmp_sol_new[node] = round(tmp_sol_new[node])
                    T_new = get_delay_from_sizes(solver, tmp_sol_new)
                    nxt_level.append((T_new, tmp_sol_new))
                    continue
                # else explore both floor and ceil
                # floor

                tmp_sol_new[node] = math.floor(tmp_sol_new[node])
                T_new = get_delay_from_sizes(solver, tmp_sol_new)
                nxt_level.append((T_new, tmp_sol_new))

                # ceil

                tmp_sol_new[node] += 1
                T_new = get_delay_from_sizes(solver, tmp_sol_new)
                nxt_level.append((T_new, tmp_sol_new))

            curr_level = nxt_level

    # get the best solution from the curr_level_dict
    curr_level = sorted(curr_level, key = lambda x: x[0])
    best_sol_T = curr_level[0][0]
    best_sol_dict = curr_level[0][1]

    elapsed = time.time() - start_time
    print(f"\nTime Taken for discretization using Branch and Bound: {round(elapsed, 4)} sec.\n")
    print(f"--- Beginning final tweaks. Local Refinement steps ---")
    start_time = time.time()

    # 2 passes through final tweaks, once reversed, once normal
    best_sol_dict, best_sol_T = final_tweaks(solver, best_sol_dict, best_sol_T, reversed_flag = True)
    best_sol_dict, best_sol_T = final_tweaks(solver, best_sol_dict, best_sol_T)

    elapsed = time.time() - start_time
    print(f"\nTime Taken for final tweaks: {round(elapsed, 4)} sec.\n")

    best_sol_sizes = 0
    for key in best_sol_dict:
        best_sol_sizes += best_sol_dict[key]


    print("--- Discrete Solution Details; Type: (Branch and Bound -BFS-) ---")
    print(f"\033[1mT_delay: {(best_sol_T):.3f}\033[0m")
    print(f"Sum of sizes: {best_sol_sizes:.2f}")

    # REDUCING THE AREA (as long as timing spec is not violated)
    print("--- Optimizing Area ---"); start_time = time.time()
    ### sizes_dict, total_size, T_delay

    area_optimal_sol = min_area_ctrl(solver, best_sol_dict)

    print(f"\033[1mT_delay: {(area_optimal_sol[2]):.3f}\033[0m")
    print(f"Sum of sizes: {area_optimal_sol[1]:.2f}")
    elapsed = time.time() - start_time
    print(f"\nTime Taken for area optimization: {round(elapsed, 4)} sec.\n")

    return area_optimal_sol[2], area_optimal_sol[0]


# locally adjusts the gate sizes to minimize the delay till it satisfies the timing specification
def final_tweaks(solver, tmp_sol, T_delay, reversed_flag = False):
    """
    Performs local refinement of a discrete gate sizing solution to reduce delay.

    This function attempts to slightly increase or decrease gate sizes to improve
    the overall circuit delay while satisfying the timing specification. It performs
    two passes (one forward, one reverse) when called twice with `reversed_flag`.

    Args:
        solver: An object containing circuit details and timing specification.
        tmp_sol (dict): Current dictionary of gate sizes.
        T_delay (float): Current delay of the circuit for the given sizing.
        reversed_flag (bool): Whether to iterate over the gates in reverse order.

    Returns:
        Tuple[dict, float]: Refined gate size dictionary and the updated delay.
    """
    t_spec = solver.solution['variables']['T0'] - 1 # timing specification
    if(reversed_flag):
        for node in reversed(tmp_sol):
            if(T_delay <= t_spec):
                break
            best_sz = tmp_sol[node]
            # tmp_sol_new = tmp_sol.copy()
            loops = 0
            while(tmp_sol[node] != 1 and loops != 2):
                tmp_sol[node] -= 1
                T_new = get_delay_from_sizes(solver, tmp_sol)
                if(T_new <= T_delay):
                    T_delay = T_new
                    best_sz = tmp_sol[node]
                    # tmp_sol = tmp_sol_new.copy()
                loops += 1
            tmp_sol[node] += loops
            loops = 0
            while(tmp_sol[node] != solver.max_gate_size and loops != 2):
                tmp_sol[node] += 1
                T_new = get_delay_from_sizes(solver, tmp_sol)
                if(T_new <= T_delay):
                    T_delay = T_new
                    best_sz = tmp_sol[node]
                    # tmp_sol = tmp_sol_new.copy()
                loops += 1
            tmp_sol[node] = best_sz
    else:

        for node in tmp_sol:
            if(T_delay <= t_spec):
                break
            best_sz = tmp_sol[node]
            # tmp_sol_new = tmp_sol.copy()
            loops = 0
            while(tmp_sol[node] != 1 and loops != 2):
                tmp_sol[node] -= 1
                T_new = get_delay_from_sizes(solver, tmp_sol)
                if(T_new <= T_delay):
                    T_delay = T_new
                    best_sz = tmp_sol[node]
                    # tmp_sol = tmp_sol_new.copy()
                loops += 1
            tmp_sol[node] += loops
            loops = 0
            while(tmp_sol[node] != solver.max_gate_size and loops != 2):
                tmp_sol[node] += 1
                T_new = get_delay_from_sizes(solver, tmp_sol)
                if(T_new <= T_delay):
                    T_delay = T_new
                    best_sz = tmp_sol[node]
                    # tmp_sol = tmp_sol_new.copy()
                loops += 1
            tmp_sol[node] = best_sz

    return tmp_sol, T_delay

#### <b>4.3. Area-optimized solution (Phase 2)</b>
- We need to get optimal area solution satisfying the given timing specification.
- The minimum area solution for a given timing specification is obtained by solving the convex optimization problem, which is continuous.
- It is required to obtain a discrete solution, which satisfies the timing constraint without very high area penalty.

In [47]:
def area_clipper(solver, sizes_sol, permit):
    """
    Reduces gate sizes to minimize area while allowing a timing penalty
    of up to 'permit' units.

    Parameters:
        solver     : the solver object containing graph and solution info
        sizes_sol  : dict mapping each gate to its current size
        permit     : maximum allowed increase in delay (relative to initial T_curr)

    Returns:
        T_curr     : new total delay after clipping
        sizes_sol  : updated size solution
    """
    tmp_G = solver.G
    Tspec = solver.solution['variables']['T0'] - 1  # timing constraint
    T_curr = get_delay_from_sizes(solver, sizes_sol)

    for node in tmp_G.nodes:
        if tmp_G.nodes[node]['type'] == 'input':
            continue
        if sizes_sol[node] > 1:
            sizes_sol[node] -= 1
            T_tmp = get_delay_from_sizes(solver, sizes_sol)
            if T_tmp <= Tspec and (T_tmp - T_curr) < permit:
                T_curr = T_tmp
            else:
                sizes_sol[node] += 1  # revert

    return T_curr, sizes_sol

def min_area_ctrl(solver, sizes_sol):
    """
    Iteratively reduces the total gate area while ensuring the timing
    specification is met or minimally violated (within a permitted range).

    Parameters:
        solver     : the solver object containing the graph and solution info
        sizes_sol  : initial gate size assignment (dictionary)

    Returns:
        area_optimal : dictionary with optimized gate sizes
        sum_sizes    : total area (sum of gate sizes)
        T_curr       : final total delay after size reductions
    """
    tmp_G = solver.G
    Tspec = solver.solution['variables']['T0'] - 1
    T_curr = get_delay_from_sizes(solver, sizes_sol)

    area_optimal = sizes_sol.copy()

    # Compute initial permitted slack margin
    permit = round((Tspec - T_curr) / 100, 4)
    if Tspec < T_curr:
        permit = 0  # T_delay already more that Tspec. (only reduce the sizes with no penalty in delay)

    # Case 1: only reduce sizes if delay does not increase at all
    if permit == 0:
        while True:
            T_tmp, area_optimal, clipped = area_clipper(solver, area_optimal, permit=0)
            if T_tmp == T_curr:
                break
            T_curr = T_tmp

    # Case 2: gradually increase permit to find the sweet spot
    while permit < (Tspec - T_curr):
        T_tmp, area_optimal= area_clipper(solver, area_optimal, permit)
        if T_tmp == T_curr:
            permit *= 2
        else:
            T_curr = T_tmp
    while (permit != 0):
        T_tmp, area_optimal= area_clipper(solver, area_optimal, permit)
        if T_tmp == T_curr:
            permit = round(permit/2, 4)
        else:
            T_curr = T_tmp

    # Compute final area
    sum_sizes = sum(area_optimal[node]
                    for node in tmp_G.nodes
                    if tmp_G.nodes[node]['type'] != 'input')

    return area_optimal, sum_sizes, T_curr


#### <b>4.4. Main Execution Block</b>

- Get the continuous solution for a given Tspec.
- Discretize the results, and get the area-optimized solution.
- Log the results in google drive.

In [53]:
def remove_ansi_codes(text):
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    return ansi_escape.sub('', text)

# Tee-like dual writer: terminal (with ANSI) + file (clean)
class DualWriter:
    def __init__(self, file_stream):
        self.terminal = sys.stdout
        self.logfile = file_stream

    def write(self, message):
        self.terminal.write(message)              # show formatted output in Colab
        self.logfile.write(remove_ansi_codes(message))  # save clean output to file

    def flush(self):
        self.terminal.flush()
        self.logfile.flush()

def run_and_log(ckt, solver, srch_idx=8, write=False):
    """
    ============================================================
    Code Snippet 6: Full Discrete Optimization and Logging Function
    ============================================================
    Description:
    This function performs the complete discrete gate sizing workflow for a given circuit.
    It first evaluates heuristic rounding strategies, then applies the Branch-and-Bound (BnB)
    algorithm to obtain an area-optimal discrete solution. If `write=True`, it logs the results
    and the discrete sizing table to files. Returns the final gate size assignment.
    ============================================================
    """
    folder_path = folder_path_discrete
    os.makedirs(folder_path, exist_ok=True)

    # Perform the optimization and logging
    log_file_path = os.path.join(folder_path, f"{ckt}_log.txt")
    size_file_path = os.path.join(folder_path, f"{ckt}_size.txt")

    if write:
        with open(log_file_path, 'w') as f:  # displays the results summary
            writer = DualWriter(f)
            with contextlib.redirect_stdout(writer):
                discrete_solutions = _get_heuristic_discretization(solver)
                start_time = time.time()
                BnB_solution = branch_nd_bound(solver, srch_idx)
                elapsed = time.time() - start_time
                print(f"Time Taken to obtain area-optimal solution: {round(elapsed, 4)} sec.\n")
    else:
        # Still compute, just no file writing or log redirection
        discrete_solutions = _get_heuristic_discretization(solver)
        start_time = time.time()
        BnB_solution = branch_nd_bound(solver, srch_idx)
        elapsed = time.time() - start_time
        print(f"[{ckt}] Time Taken to obtain area-optimal solution: {round(elapsed, 4)} sec.")

    # Prepare tabular data
    table_data = []
    for i, (key, value) in enumerate(BnB_solution[1].items(), 1):
        gate_name = solver.G.nodes[key].get('name', 'UNKNOWN')
        continuous_size = round(solver.solution['variables']['x'][solver.G.nodes[key].get('g_idx')], 3)
        table_data.append([i, key, gate_name, continuous_size, value])

    headers = ["S. No.", "Node Name", "Gate Name", "Continuous Size", "Discrete Size"]

    if write:
        with open(size_file_path, 'w') as f:
            f.write(f"Circuit: {ckt}; (Tspec/Twall) = {solver.relax_factor}\n")
            f.write(tabulate(table_data, headers=headers, tablefmt="grid"))
            f.write(f"\n\nT_delay (continuous): {(solver.solution['variables']['T0'] - 1):.3f}")
            f.write(f"\nT_delay (discrete): {BnB_solution[0]:.3f}")

    return BnB_solution[1]

# Get Discrete solutions
# ckt_list = ["c2670", "c3540", "c5315", "c6288", "c7552"]
ckt_list = ["c17", "c432", "c880", "c1908"]

for ckt in ckt_list:
    solver = get_solution(ckt, relax_factor=1.3)
    discrete_sizes = run_and_log(ckt, solver, srch_idx = 4, write = True)

# ## Code block to check if the input capacitance does not exceed the defined limit
# # For each primary input, limit the total load it drives (sum of le * size of each gate it connects to)
# loading_ok = True
# for node, attr in solver.G.nodes(data=True):
#     if attr.get('type') == 'input':
#         total_load = 0
#         for succ in solver.G.successors(node):  # all gates driven by this input
#             gate_type, gate_idx = solver.gate_info_map[solver.G.nodes[succ]['name']]
#             le = solver.le_dict[gate_type]
#             total_load += le * discrete_sizes[succ]
#         if(total_load > 50):
#             print("\n\033[91m--- Input load condition violated ---\033[01m\n")
#             print(f"Node: {node}, Total load = {total_load}; Size reduction required")
#             loading_ok = False
#             break

# if(loading_ok):
#     print("\033[92m--- Input load condition satisfied ---\033[01m")



<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)
<ipython-input-47-ec3c5dbb6312>:80: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  sum_sizes = sum(area_optimal[node]
<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)


Using solver 'mosek_cli'
 for 24 free variables
  in 48 posynomial inequalities.
Solving took 0.0187 seconds.
Total elapsed time (including setup): 0.0245 seconds
Using solver 'mosek_cli'
 for 24 free variables
  in 49 posynomial inequalities.
Solving took 0.0172 seconds.
Total elapsed time (including setup): 0.0238 seconds
--- Continuous Solution Details ---
ISCAS85 Benchmark ckt: c17
T_delay: 37.482 --->  [Optimal Area Solution, (T_spec/T_wall) = 1.3]
Sum of sizes: 136.11

--- Discrete Solution Details; Type: (floor) ---
T_delay: 38.222
Sum of sizes: 134.00

--- Discrete Solution Details; Type: (ceil) ---
T_delay: 36.954
Sum of sizes: 140.00

--- Discrete Solution Details; Type: (round) ---
T_delay: 38.222
Sum of sizes: 135.00

--- Discrete Solution Details; Type: (continuous) ---
T_delay: 37.482
Sum of sizes: 136.11

--- Beginning discretization for c17; T_spec = T_wall*1.3 ---

Time Taken for discretization using Branch and Bound: 0.002 sec.

--- Beginning final tweaks. Local Refin

<ipython-input-47-ec3c5dbb6312>:80: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  sum_sizes = sum(area_optimal[node]


T_delay: 199.484
Sum of sizes: 482.00

Time Taken for area optimization: 1.4699 sec.

Time Taken to obtain area-optimal solution: 2.5356 sec.



<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)


Using solver 'mosek_cli'
 for 1850 free variables
  in 2897 posynomial inequalities.
Solving took 0.784 seconds.
Total elapsed time (including setup): 2.3424 seconds
Using solver 'mosek_cli'
 for 1850 free variables
  in 2898 posynomial inequalities.
Solving took 0.517 seconds.
Total elapsed time (including setup): 3.1079 seconds
--- Continuous Solution Details ---
ISCAS85 Benchmark ckt: c880
T_delay: 154.607 --->  [Optimal Area Solution, (T_spec/T_wall) = 1.3]
Sum of sizes: 891.27

--- Discrete Solution Details; Type: (floor) ---
T_delay: 167.857
Sum of sizes: 862.00

--- Discrete Solution Details; Type: (ceil) ---
T_delay: 162.228
Sum of sizes: 1417.00

--- Discrete Solution Details; Type: (round) ---
T_delay: 163.524
Sum of sizes: 888.00

--- Discrete Solution Details; Type: (continuous) ---
T_delay: 154.607
Sum of sizes: 891.27

--- Beginning discretization for c880; T_spec = T_wall*1.3 ---

Time Taken for discretization using Branch and Bound: 5.1727 sec.

--- Beginning final twea

... (2 outputs left out)

<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)


Using solver 'mosek_cli'
 for 3885 free variables
  in 5617 posynomial inequalities.
Solving took 1.27 seconds.
Total elapsed time (including setup): 8.2032 seconds
Using solver 'mosek_cli'
 for 3885 free variables
  in 5618 posynomial inequalities.
Solving took 1.22 seconds.
Total elapsed time (including setup): 7.8867 seconds
--- Continuous Solution Details ---
ISCAS85 Benchmark ckt: c1908
T_delay: 230.860 --->  [Optimal Area Solution, (T_spec/T_wall) = 1.3]
Sum of sizes: 1531.48

--- Discrete Solution Details; Type: (floor) ---
T_delay: 246.889
Sum of sizes: 1496.00

--- Discrete Solution Details; Type: (ceil) ---
T_delay: 235.639
Sum of sizes: 2640.00

--- Discrete Solution Details; Type: (round) ---
T_delay: 239.409
Sum of sizes: 1511.00

--- Discrete Solution Details; Type: (continuous) ---
T_delay: 230.860
Sum of sizes: 1531.48

--- Beginning discretization for c1908; T_spec = T_wall*1.3 ---

Time Taken for discretization using Branch and Bound: 33.3293 sec.

--- Beginning final

<ipython-input-47-ec3c5dbb6312>:80: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  sum_sizes = sum(area_optimal[node]


T_delay: 230.817
Sum of sizes: 1572.00

Time Taken for area optimization: 25.2388 sec.

Time Taken to obtain area-optimal solution: 62.1286 sec.



### <b>5. Drive strength determination</b>
- From the sizes obtained after discretization, it can be decided, what sizes of gates can be a part of the standard cell design library.
- First get the consolidated data of the gate_type and count from the circuits in the benchmark suite.
- Using that data, decide upon the sizes that need to be part of the standard cell library.

In [36]:
def build_size_dict(solver, discrete_sizes, sizes_dict=None):
    """
    Groups the gate sizes from a given discrete solution by gate type.
    It returns a dictionary mapping each gate type to a list of sizes used in the circuit.
    """
    if sizes_dict is None:
        sizes_dict = {}

    tmp_G = solver.G
    std_cell_sizes = sizes_dict

    for node in tmp_G.nodes:
        if tmp_G.nodes[node]['type'] != 'input':
            gate_type = tmp_G.nodes[node]['type']
            std_cell_sizes.setdefault(gate_type, []).append(discrete_sizes[node])

    return std_cell_sizes

# # ONE-TIME EXECUTION BLOCK

# # Get Discrete solutions
# # Get the consolidated sizes dictionary from the discrete solution of every circuit

# ckt_list = ["c17", "c432", "c880", "c1908", "c2670", "c3540", "c5315", "c6288", "c7552"]
# sizes_dict = {}
# for ckt in ckt_list:
#     # size dictionary can be made to store consolidated database of discrete sizes for different relax factors
#     solver = get_solution(ckt, relax_factor=1.3)
#     discrete_sizes = run_and_log(ckt, solver, srch_idx = 4)
#     sizes_dict = build_size_dict(solver, discrete_sizes, sizes_dict)

# # temporary (dump the sizes dictionary for future use)==========================
# # this dictionary can be used to get the drive strength of the cells in the standard library


# file_name = "sizes_dict.json"

# sizes_dict_path = os.path.join(sizes_dict_path, file_name)
# with open(sizes_dict_path, "w") as f:
#     json.dump(sizes_dict, f, indent=2)  # indent=2 for pretty formatting


In [41]:
# include the gates, that have the highest count in the intervals of 4
# include the gates, if they account for more than a threshold percentage

file_name = "sizes_dict.json"

with open(os.path.join(sizes_dict_path, file_name), "r") as f:
    sizes_dict = json.load(f)

def get_std_lib(cell_sizes):
    std_cell_lib = {}
    count = 0

    for gate_type in cell_sizes:

        std_cell_lib[gate_type] = []  # array to store the sizes to part of the std lib
        size_count = np.zeros(solver.max_gate_size) # array to store the number of occurences of each gate type

        num_gates = len(cell_sizes[gate_type]) # total number of gates of current type
        count += num_gates

        for size in cell_sizes[gate_type]:
            size_count[size-1] += 1
        for i in range(len(size_count)):
            fraction = size_count[i]/num_gates

            if fraction >= 0.001: # threshold percentage (0.5%)
                std_cell_lib[gate_type].append(i+1)

            if(i%4 == 3):
                j = 0
                curr_max = 0
                included = False
                while(j <= 3):
                    fraction = size_count[i-j]/num_gates
                    if fraction >= 0.001: # a gate is already included from the present interval
                        included = True
                        break
                    j += 1

                if(not included):
                    # no gate from the current interval is included
                    maxx = max(size_count[i-3:i+1])
                    if(maxx == 0):
                        continue
                    # include the smallest gate from the interval (emphasizing on area optimality)
                    for k in range(i-3, i+1):
                        if size_count[k] == maxx:
                            std_cell_lib[gate_type].append(k+1)
                            break
                else:
                    continue
            else:
                continue

    return std_cell_lib

get_std_lib(sizes_dict)

{'NAND_2': [1, 2, 3, 4, 5, 8, 11, 12, 13, 17, 46],
 'NOT': [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 20, 22, 27, 29],
 'NAND_4': [1, 2, 3, 4, 6, 8, 9, 11, 23, 25],
 'NOR_2': [1, 2, 3, 4, 5, 6, 9, 13, 18, 27],
 'NAND_3': [1, 2, 3, 4],
 'NOR_3': [1, 2, 4, 8],
 'NOR_4': [1, 2]}

### <b>Phase 2: CVXPY Mixed-Integer GP Solver</b>

#### <b>Ph2.1. Gate Sizing Model Class</b>
- Declares all necessary variables for the gate sizing optimization problem.
- Builds parasitic delay and logical effort dictionaries based on the circuit structure.
- Discrete sizing introduces a `log_size` variable (used in delay constraints) and a `g_size` variable (which will be discretized).
- Gate delay constraints are formulated in terms of the `log_size` variable to retain convexity.
- The objective function minimizes the sum of `g_size` and includes a penalty term: the absolute difference between `exp(log_size)` and `g_size`.
- This encourages `exp(log_size)` to converge towards an integer value.
- After optimization, `exp(log_size)` can be rounded to the nearest valid gate size to produce a discrete solution.




In [56]:
# Gate Sizing Model class for mixed integer GP formulation
class cp_GateSizingModel:
    def __init__(self, G, gate_info_map, max_gate_size, out_cap, max_in_cap):
        """
        Initialize vector variables, delay parameters, and other constants for gate sizing.
        """
        self.G = G
        self.gate_info_map = gate_info_map
        self.n_gates = len(gate_info_map)
        self.n_nodes = len(G.nodes)
        self.constraints = []

        # One hot encoding the gate sizes
        self.size_sel = [cp.Variable((max_gate_size), boolean = True) for _ in range(self.n_gates)]
        # Integer gate_size
        self.g_size = cp.Variable(self.n_gates, integer = True)
        # log_sizes
        self.log_size = cp.Variable(self.n_gates, pos = True)

        self.delta1 = cp.Variable(self.n_gates)

        self.constraints += [
            self.delta1 >= cp.exp(self.log_size) - self.g_size,
        ]

        # possible sizes
        possible_sizes = np.arange(1, max_gate_size + 1, dtype = float)
        possible_logs = np.log(possible_sizes)

        # preliminary constraints
        for i in range(self.n_gates):

            # self.constraints.append(cp.sum(self.size_sel[i]) == 1)

            self.constraints.append(self.g_size[i] >= 1)
            self.constraints.append(self.g_size[i] <= max_gate_size)

            # self.constraints.append(self.g_size[i] == cp.sum(cp.multiply(possible_sizes, self.size_sel[i])))
            # self.constraints.append(self.log_size[i] == cp.sum(cp.multiply(possible_logs, self.size_sel[i])))

        # Arrival time at all circuit nodes (logarithmic domains)
        self.a_t = cp.Variable(self.n_nodes, pos = True)

        # T0 represents the overall circuit delay (i.e., Twall)
        self.T0 = cp.Variable(pos = True)

        # Delay models
        self.pd_dict = self._init_parasitic_delays()
        self.le_dict = self._init_logical_efforts()

        # Constants
        self.out_cap = out_cap
        self.max_in_cap = max_in_cap
        self.max_gate_size = max_gate_size

    def _init_parasitic_delays(self):
        """
        Returns a dictionary of parasitic delays for each gate type.
        """
        pd_dict = {"NOT": 1}
        for i in range(2, 5):
            pd_dict[f"NAND_{i}"] = i
            pd_dict[f"NOR_{i}"] = i
        return pd_dict

    def _init_logical_efforts(self):
        """
        Returns a dictionary of logical efforts for each gate type.
        """
        le_dict = {"NOT": 1}
        for i in range(2, 5):
            le_dict[f"NAND_{i}"] = (i + 2) / 3
            le_dict[f"NOR_{i}"] = (2 * i + 1) / 3
        return le_dict

    def get_variables(self):
        """
        Return all GP variables and constants as a dictionary.
        """
        return {
            'g_size': self.log_size,
            'a_t': self.a_t_log,
            'T0': self.T0,
            'pd_dict': self.pd_dict,
            'le_dict': self.le_dict,
            'out_cap': self.out_cap,
            'max_in_cap': self.max_in_cap,
            'max_gate_size': self.max_gate_size
        }

#### <b>Ph2.2. Constraints Builder Class</b>
- Sets up the constraints of mixed-integer optimization problem

In [43]:
class cp_ConstraintBuilder:
    def __init__(self, G,
                 size_sel, g_size, log_size,a_t, T0,
                 gate_info_map, le_dict, pd_dict, constraints,
                 out_cap, max_gate_size, max_in_cap):
        self.G = G

        # handling the gate size
        self.size_sel = size_sel
        self.g_size = g_size
        self.log_size = log_size

        self.T0 = T0
        self.a_t = a_t
        self.gate_info_map = gate_info_map
        self.le_dict = le_dict
        self.pd_dict = pd_dict
        self.out_cap = out_cap
        self.max_gate_size = max_gate_size
        self.max_in_cap = max_in_cap
        self.constraints = constraints

    # GATE SIZE CONSTRAINTS
    def gate_size_constraints(self):

        # (size should be between the allowed limits) (1 <= size <= max_gate_size)
        # inheretly taken care of by the sum(size_sel) = 1 constraint

# ==============================================================================
        # For each primary input, limit the total load it drives (sum of le * size of each gate it connects to)
        # --- INPUT GATE SIZE HANDLING ---

        for node, attr in self.G.nodes(data=True):
            if attr.get('type') == 'input':
                total_load_terms = []
                for succ in self.G.successors(node):  # all gates driven by this input
                    gate_type, gate_idx = self.gate_info_map[self.G.nodes[succ]['name']]
                    le = self.le_dict[gate_type]
                    # Use log-domain formulation: exp(log(size) + log(le)) = le * size
                    total_load_terms.append(cp.exp(self.log_size[gate_idx] + np.log(le)))
                if total_load_terms:
                    self.constraints.append(cp.sum(total_load_terms) <= self.max_in_cap)

# ==============================================================================
    # --- TIMING CONSTRAINTS (I/O TIMINGS) ---
    # Same code works for logarithmic arrival times

    def timing_constraints(self):
        for node, attr in self.G.nodes(data=True):
            # Set the arrival time at INPUTS = 0
            if attr.get('type') == 'input':
                self.constraints.append(self.a_t[attr['n_idx']] == 1)
                continue

            # Set T0 >= arrival time at OUTPUTS
            if len(list(self.G.successors(node))) == 0:
                self.constraints.append(self.T0 >= self.a_t[attr['n_idx']])

# ==============================================================================

    # (ADDING GATE DELAYS TO TIMING CONSTRAINTS)
    def delay_constraints(self):
        G = self.G
        for node, attr in G.nodes(data=True):
            curr_gate = attr.get('name')

            if attr.get('type') != 'input':  # current node is the output of a gate
                pd = self.pd_dict[G.nodes[node]['type']]

                # directly getting the required fan-in and fan-out lists from the graph
                fan_in_nodes = list(G.predecessors(node))  # all fan-in nodes (by node ID)
                fan_out_gates = [G.nodes[succ]['name'] for succ in G.successors(node) if G.nodes[succ]['type'] != 'input']
                fo_idx_list = []; le_list = []
                for gate in fan_out_gates:
                    fo_idx_list.append(self.gate_info_map[gate][1])
                    le_list.append(self.le_dict[self.gate_info_map[gate][0]])

                fo_len = len(fan_out_gates)

                if not fan_out_gates:
                    # a primary output, with no gates connected to it (only driving the output capacitance)
                    if attr.get('is_output', False):
                        log_sum_expr = cp.log_sum_exp(
                            cp.hstack([np.log(self.out_cap)])
                        )

                        for fi_node in fan_in_nodes:
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd + cp.exp(log_sum_expr - self.log_size[self.gate_info_map[curr_gate][1]]))
                    else:
                        for fi_node in fan_in_nodes:
                            # no fan-out gates, and not a primary output (FLOATING GATE). (In this case, only parasitic delay is there)
                            self.constraints.append(self.a_t[attr['n_idx']] >= self.a_t[G.nodes[fi_node]['n_idx']] + pd)

                else:
                    for fi_node in fan_in_nodes:
                        if attr.get('is_output', False):
                            log_sum_expr = cp.log_sum_exp(
                                cp.hstack([self.log_size[fo_idx_list[i]] + np.log(le) for i, le in enumerate(le_list)] + [np.log(self.out_cap)])
                            )
                        else:
                            log_sum_expr = cp.log_sum_exp(
                                cp.hstack([self.log_size[fo_idx_list[i]] + np.log(le) for i, le in enumerate(le_list)])
                            )

                        pred_idx = G.nodes[fi_node]['n_idx']
                        curr_idx = attr['n_idx']
                        gate_idx = self.gate_info_map[curr_gate][1]

                        self.constraints.append(self.a_t[curr_idx] >= self.a_t[pred_idx] + pd + cp.exp(log_sum_expr - self.log_size[gate_idx]))

    def build_constraints(self):

        self.gate_size_constraints()
        self.timing_constraints()
        self.delay_constraints()
        return self.constraints


#### <b>Ph2.3. Solver Class</b>
- Sets up the problem, defines the objective function, and solves the mixed integer optimization problem.

In [44]:
class cp_Solver:
    def __init__(self, ckt_name, CircuitGraphClass, GateSizingModelClass, ConstraintBuilderClass, constants):
        self.ckt_name = ckt_name
        self.CircuitGraphClass = CircuitGraphClass
        self.GateSizingModelClass = GateSizingModelClass
        self.ConstraintBuilderClass = ConstraintBuilderClass

        self.out_cap, self.max_in_cap, self.max_gate_size, self.Tspec, self.relax_factor = constants

        self.sizes = None
        self.Twall = None

        self._initialize()

    def _initialize(self):
        # Build the circuit graph and extract basic data
        self.solution = None
        self.ckt = self.CircuitGraphClass(self.ckt_name)
        self.G = self.ckt.get_graph()
        self.gate_info_map = self.ckt.get_gate_info_map()

        # Initialize sizing model

        # class cp_GateSizingModel:
        #     def __init__(self, G, gate_info_map, max_gate_size, out_cap, max_in_cap):


        self.sizing_model = self.GateSizingModelClass(self.G, self.gate_info_map, self.max_gate_size, self.out_cap, self.max_in_cap)

        self.size_sel = self.sizing_model.size_sel
        self.g_size = self.sizing_model.g_size
        # # --- LOG SIZES ---
        self.log_size = self.sizing_model.log_size
        self.delta1 = self.sizing_model.delta1
        # self.a_t = self.sizing_model.a_t
        # # --- LOG ARRIVAL TIMES ---
        self.a_t = self.sizing_model.a_t

        self.T0 = self.sizing_model.T0
        self.le_dict = self.sizing_model.le_dict
        self.pd_dict = self.sizing_model.pd_dict
        self.constraints = self.sizing_model.constraints

        # class cp_ConstraintBuilder:
        #     def __init__(self, G,
        #                  size_sel, g_size, log_size, a_t, T0,
        #                  gate_info_map, le_dict, pd_dict, constraints,
        #                  out_cap, max_gate_size, max_in_cap):

        # BUILD CONSTRAINTS ====================================================
        self.constraints = self.ConstraintBuilderClass(
            self.G,
            # ---variables associated with gate size---
            self.size_sel, self.g_size, self.log_size,
            # ---variables associated with arrival times---
            self.a_t,
            # ------
            self.T0, self.gate_info_map, self.le_dict, self.pd_dict,
            self.constraints, self.out_cap, self.max_gate_size, self.max_in_cap
        ).build_constraints()

    def solve(self):
        """
        Solves the optimization problem and returns key results.
        """
        # outputs = []
        # for node, attr in self.G.nodes(data=True):
        #     if attr.get('is_output', False):
        #         outputs.append(attr['n_idx'])

        # minimize the area with a certain output timing specification

        objective = cp.Minimize(cp.sum(self.g_size) + 1000*cp.sum(cp.square(self.delta1)))

        self.constraints.append(self.T0 <= self.Tspec)

        prob = cp.Problem(objective, self.constraints)
        start_time = time.time()
        self.solution = prob.solve(solver = cp.MOSEK)
        elapsed_time = time.time() - start_time
        print("--- DISCRETE SOLUTION ---")
        # print("Total Gate Size:", cp.sum(self.g_size).value)
        # print("Critical Path Delay (T0):", self.T0.value - 1)
        print(f"Total elapsed time (including setup): {elapsed_time:.4f} seconds")

        sizes_dict = {}
        gate_idx = 0
        for node in self.G.nodes:

            if(self.G.nodes[node]['type'] != 'input'):
                sizes_dict[node] = round(np.exp(self.log_size[gate_idx].value))
                gate_idx += 1

        return sizes_dict

#### <b>Ph2.4. Main Execution Block</b>
- Gets the continuous solution, which helps in defining the Tspec in terms of Twall.
- Then instantiates a discrete solver class, and provides the appropriate Tspec to get the discrete area-minimized solution.

In [57]:
sys.setrecursionlimit(100000)

### RESULT DUMP CODE BLOCK======================================================

sizing_constants = (1000, 50, 64)  # (out_cap, max_in_cap, max_gate_size) (Normalized to 1x inverter size)
ckt = "c17"
folder_path = f"{path_continuous}/{ckt}"
# Relaxed timing constraint (Tspec = Twall * relax_factor) (AREA+OPTIMIZATION)
relax_factor = 1.5
relax_factor_arr = [1.3]

# Solver for Twall
solver = TSolver(
    ckt_name=ckt,
    CircuitGraphClass=CircuitGraph,
    GateSizingModelClass=GateSizingModel,
    ConstraintBuilderClass=ConstraintBuilder,
    constants=sizing_constants
)

solver.solve()
Twall = solver.solution["variables"]["T0"]
sizes = list(solver.solution["variables"]["x"])
sizes_rounded = ['%.3f' % val for val in sizes]
sum_sizes = sum(sizes)


print(f"\n\033[1mTwall = {(Twall - 1):.3f}\033[0m\n")
print("Sizes of gates =", sizes_rounded)
print("Sum of sizes:", round(sum_sizes, 2))
# Saving sizing results
save_sizing_results(solver, folder_path=folder_path, relax_factor=relax_factor)

print("===================================================================================")

for relax_factor in relax_factor_arr:
    # solver for Opimal area, given a timing specification
    sizing_constants_A = sizing_constants + (relax_factor*(Twall - 1) + 1, relax_factor)
    solver_A = cp_Solver(
        ckt_name=ckt,
        CircuitGraphClass=CircuitGraph,
        GateSizingModelClass=cp_GateSizingModel,
        ConstraintBuilderClass=cp_ConstraintBuilder,
        constants=sizing_constants_A
    )

    discrete_sizes_dict = solver_A.solve()
    # print(discrete_sizes_dict)
    # print(solver_A.g_size.value)

    sum_of_sizes = 0
    for key, value in discrete_sizes_dict.items():
        sum_of_sizes += value
    print("\nSum of sizes:", sum_of_sizes)
    print("Verification (discrete_sizes_delay):", get_delay_from_sizes(solver, discrete_sizes_dict))
    print("Tspec = ", round(relax_factor*(Twall - 1), 3))

    print("g_size: ", solver_A.g_size.value)
    print("log_size: ", np.exp(solver_A.log_size.value))

<ipython-input-27-da361fe365d5>:76: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  objective = self.T0 + 0.0001*sum(self.a_t[i] for i in outputs)


Using solver 'mosek_cli'
 for 24 free variables
  in 48 posynomial inequalities.
Solving took 0.0197 seconds.
Total elapsed time (including setup): 0.0272 seconds

Twall = 28.833

Sizes of gates = ['9.268', '28.232', '37.500', '18.750', '64.000', '64.000']
Sum of sizes: 221.75
Results saved to: /content/drive/MyDrive/EE6332/sizing_results/c17/c17_minT_.txt

--- DISCRETE SOLUTION ---
Total elapsed time (including setup): 0.3443 seconds

Sum of sizes: 137
Verification (discrete_sizes_delay): 37.46031746031746
Tspec =  37.482
g_size:  [ 6. 14. 18.  9. 45. 45.]
log_size:  [ 5.89469505 13.99679312 17.99349269  8.99595332 44.97544951 44.98653178]
